In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *

In [2]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/solo_L1_phi-fdt-ilam_20241120T001503_V202412020209C_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz']

In [3]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')

xd = s['xd']
yd = s['yd']
xu = s['xu']
yu = s['yu']

In [177]:
file_hmi = files[0]
file_fdt = files[2]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

nx, ny = header_fdt['NAXIS2'], header_fdt['NAXIS1']
x0, y0 = header_fdt['PXBEG2'] - 1, header_fdt['PXBEG1'] - 1
data_fdt = bilinear(data_fdt, xd[x0:x0+nx, y0:y0+ny] - x0, yd[x0:x0+nx, y0:y0+ny] - y0)

data_hmi = rebin(data_hmi, 8, update_header=header_hmi)
data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=True, mu_thr=0.2, crota=header_fdt['CROTA'] + 0.3, rsun=330.25 + 0.5, xc=421.64, yc=457.87)

In [178]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt * 1.25 - data_hmi, cmap='seismic', vmin=-50, vmax=50)
plt.tight_layout()

In [138]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, cmap='seismic', vmin=-50, vmax=50)
plt.tight_layout()

In [180]:
plt.figure(figsize=(10,10))
plt.scatter(data_hmi, data_fdt, s=0.01)
plt.xlim(-200,200)
plt.ylim(-200,200)

(-200.0, 200.0)

In [179]:
plt.figure(figsize=(10,10))
plt.scatter(data_hmi, data_fdt, s=0.01)
plt.xlim(-20,20)
plt.ylim(-20,20)

(-20.0, 20.0)

In [147]:
dx = 0.5

x = []
y = []
for x_ in np.arange(-10,10,dx):
    a = x_ ** 3
    b = (x_ + dx) ** 3

    t = np.where(np.all([data_hmi > a, data_hmi < b], axis=0))
    x += [np.median(data_hmi[t])]
    y += [np.median(data_fdt[t])]

x = np.array(x)
y = np.array(y)

plt.figure(figsize=(10,10))
plt.scatter(data_hmi, data_fdt, s=0.01)
plt.plot(x, y, color='black')

plt.xlim(-50,50)
plt.ylim(-50,50)

#plt.grid(True)
plt.tight_layout()